# Getting started with LangChain

1. Simple LLM calls with streaming
2. Dynamic prompts template(translation app)
3. Building chains (story generator with analysis)
4. Conversational Q&A assistant with memory
5. Tool integration(calculator and weather)

In [2]:
import langchain

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# 1. simple LLM call

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [14]:
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000262BC29FED0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000262BC334910>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
#create message
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("Who is the highest paid cricketer in the world?")
]

In [20]:
#invoke the model
response = model.invoke(messages)
response

AIMessage(content='**Virat\u202fKohli** is the highest‑paid cricketer in the world.  \n\n| Year | Earnings (USD) | Notes |\n|------|----------------|-------|\n| **2024** | **≈\u202f$15.2\u202fmillion** | Top‑tier match fees, brand endorsements (e.g., Puma, Sunsilk, Gatorade), and his own brand ventures. |\n| **2023** | **≈\u202f$15.1\u202fmillion** | Same mix of match fees and endorsements. |\n\n*Source: Forbes’ annual “Highest‑Paid Cricketers” list (2024 edition).*\n\nKohli’s earnings far outpace the next tier of players (e.g., Rohit\u202fSharma, $9–10\u202fmillion), making him the clear leader in the cricketing world’s pay scale.', additional_kwargs={'reasoning_content': 'We need to answer: highest paid cricketer in the world. As of 2024-09-28, need up-to-date info. The highest paid cricketer is usually Virat Kohli? But recently, Chris Gayle? Let\'s recall: According to Forbes, in 2023, the highest paid cricketer was Virat Kohli at $15.1 million. But in 2024, maybe still Kohli or may

In [21]:
print(response.content)

**Virat Kohli** is the highest‑paid cricketer in the world.  

| Year | Earnings (USD) | Notes |
|------|----------------|-------|
| **2024** | **≈ $15.2 million** | Top‑tier match fees, brand endorsements (e.g., Puma, Sunsilk, Gatorade), and his own brand ventures. |
| **2023** | **≈ $15.1 million** | Same mix of match fees and endorsements. |

*Source: Forbes’ annual “Highest‑Paid Cricketers” list (2024 edition).*

Kohli’s earnings far outpace the next tier of players (e.g., Rohit Sharma, $9–10 million), making him the clear leader in the cricketing world’s pay scale.


In [23]:
#streaming example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

**Virat Kohli** is widely reported as the highest‑paid cricketer in the world (as of 2026).  

- **Annual earnings:** Roughly **$8 – 9 million** a year.  
- **Sources of income:**  
  - **IPL contract** with the Royal Challengers Bangalore (≈ $4 – 5 million).  
  - **National team salary** (India) and match fees (≈ $1 – 2 million).  
  - **Endorsements** (Nike, MRF, and several other global brands) (≈ $2 – 3 million).  
  - **Other business ventures** and media appearances.

Other top‑earning players—such as Rohit Sharma, Babar Azam, and Steve Smith—earn well into the millions as well, but Kohli’s combination of IPL, national contracts and endorsements keeps him at the very top of the list.

# Dynamic prompt templates

In [33]:
from langchain_core.prompts import ChatPromptTemplate

#create a translation prompt

translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. You translate the following. {text} from {source_language} to {target_language}.Maintain your tone and style of the original text."),
    ("user", "{text}"),
])

#using the template 
prompt = translation_template.invoke({
    "source_language": "En",
    "target_language": "Ma",
    "text": "How will you build a successful career in AI and ML?"
})

In [34]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. You translate the following. How will you build a successful career in AI and ML? from En to Ma.Maintain your tone and style of the original text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='How will you build a successful career in AI and ML?', additional_kwargs={}, response_metadata={})])

In [35]:
translated_response = model.invoke(prompt)
print(translated_response.content)

तुम्ही AI आणि ML मध्ये यशस्वी करिअर कसे उभारणार?


# Building first chain

In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def create_story_chain():
    ##template for story generation
    story_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a creative story teller. Write a short and engaging story. based on given theme."),
            ("user", "Theme: {theme}\n Main character: {character}\n Setting: {setting}\n"),
        ])

    #template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are literary critic. Analyze the story and provide a brief summary, key themes, and your opinion on the story."),
        ("user", "{story}"),
    ])

    story_chain=(
        story_prompt | model | StrOutputParser() | analysis_prompt | model | StrOutputParser()
    )

    #create a function to pass the story to the analysis
    def analyze_story(story_text):
        return {"story": story_text}

    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        |StrOutputParser()
    )

    return analysis_chain

In [37]:
chain = create_story_chain()
chain

{
  story: ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative story teller. Write a short and engaging story. based on given theme.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main character: {character}\n Setting: {setting}\n'), additional_kwargs={})])
         | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs':

In [39]:
chain = create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative story teller. Write a short and engaging story. based on given theme.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main character: {character}\n Setting: {setting}\n'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outpu

In [40]:
result = chain.invoke({
    "theme": "Cricket",
    "character": "A young boy named cheeku",
    "setting": "A small town in the countryside"
})

print("Story and Analysis: ")
print(result)

Story and Analysis: 
**Brief Summary (Re‑framed)**  
In the unassuming village of Sundarpur, Cheeku, a modest boy with a battered bat, yearns to play cricket for the town’s elite club. A humble mentor, Ramesh, teaches him that skill, strategy and mental focus outweigh expensive gear. When Cheeku’s rag‑tag “Green Grass” squad enters the village tournament, they upset the favored Sundarpur Cricket Club, earning the respect of the town’s establishment and proving that talent and determination can triumph over wealth and prejudice.

**Key Themes (Re‑interpreted)**  

| Theme | Narrative Manifestation | Significance |
|-------|------------------------|--------------|
| **Perseverance vs. Socio‑Economic Barriers** | Cheeku’s relentless practice with a worn bat, defying the club’s elitist gatekeeping. | Demonstrates that ambition can flourish even in constrained circumstances. |
| **Mentorship & Knowledge Transfer** | Ramesh’s counsel—“Use your mind, not just your muscle”—shapes Cheeku’s stra